In [ ]:
# Load dataset

import numpy as np
import pandas as pd
from blox2 import split_df_by_n_rows, load_features

df_features = pd.read_csv("data/material/features.csv", header=None)
df_features.head() # 132 features for each material

In [ ]:
df_properties = pd.read_csv("data/material/properties.csv")
df_properties.head() # 2 properties for each material

In [ ]:
# Setup
observed_features, unchecked_features = split_df_by_n_rows(df_features, 10)
observed_values, unchecked_values = split_df_by_n_rows(df_properties, 10)

# This can be replaced with simulations etc.
def get_true_value(id):
    return np.asarray(unchecked_values.iloc[id - len(observed_features)])

In [ ]:
# Loop

import time
from blox2 import SteinNoveltySelector
from blox2.predictor import LightGBMPredictor

n_total_suggestions = 500
report_interval = 50
sigma = 1
n_candidates_in_batch = 1 # Set to n >= 1 for batch proposals

predictor = LightGBMPredictor(n_estimators=100, use_uncertainty=True, quantiles=[0.1, 0.9])
selector = SteinNoveltySelector(
    observed_features, observed_values, unchecked_features, 
    predictor, 
    sigma = sigma, 
    use_uncertainty = True,
    uncertainty_ratio = 0.5,
    normalize_features = True, 
    n_obs_samples = None, # Set to 100~500 to use sampled Stein Discrepancy
    
    ### Comment out below to use in-batch diversity: also set n_candidates_in_batch above to >=2 ###
    # use_batch_penalty = True,
    # batch_penalty_ratio = 0.5,
    # batch_penalty_type = "simhash",
    # batch_penalty_simhash_bits = 8,
)

t0 = time.perf_counter()
n_iters = min(n_total_suggestions, len(unchecked_features)) // n_candidates_in_batch
for i in range(n_iters):
    ids = selector.next_candidates(n=n_candidates_in_batch)
    for id in ids:
        selector.observe(id, get_true_value(id))
    if (i+1) % report_interval == 0:
        print(f"Loop {i+1} finished. Passed time: {time.perf_counter() - t0}")
print(f"All suggestions are finished.")

In [ ]:
# Record results

import csv, datetime, os
import matplotlib.pyplot as plt
import pandas as pd

dir_name = "results/" + datetime.datetime.now().strftime("%m-%d_%H-%M")

os.makedirs(dir_name, exist_ok=True)
observation_history = selector.make_observation_history()

with open(os.path.join(dir_name, "candidate_histories.csv"), "w", newline="") as f:
    writer = csv.writer(f)
    for x in selector.candidate_id_history[selector.initial_n_obs:]:
        writer.writerow([x])
np.savetxt(os.path.join(dir_name, "initial_observed_properties.csv"), observation_history[:selector.initial_n_obs], delimiter=",",)
np.savetxt(os.path.join(dir_name, "observation_histories.csv"), observation_history[selector.initial_n_obs:], delimiter=",",)

In [ ]:
# Scatterplot

scatter_plot_intervals = [10, 200]

for n in scatter_plot_intervals:
    if n <= 0 or n > len(observation_history):
        continue
    data = observation_history[:n + selector.initial_n_obs + 1]

    plt.figure()
    plt.scatter(data[:, 0], data[:, 1], s=5)
    plt.xlabel(observed_values.columns[0])
    plt.ylabel(observed_values.columns[1])
    plt.title(f"Number of sampling: {n}")
    plt.savefig(os.path.join(dir_name, f"scatter_{n}.png"))
    plt.grid(True)
    plt.show()